# Symmetric Adversarial Neural Cryptography with Differential Privacy

This notebook extends the symmetric model with differential privacy to protect training data.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Concatenate, Reshape, Dense, Conv1D, Flatten
from keras.optimizers import Adam
from tqdm import tqdm

# Import differential privacy module
from differential_privacy import (
    DPOptimizer,
    PrivacyAccountant,
    get_privacy_preset,
    PRIVACY_PRESETS
)

## Configuration

Set model and privacy parameters.

In [ ]:
# Model parameters
p_len = 16
k_len = 16
c_len = 16
epochs = 20
loss_threshold = 0.1
learning_rate = 0.0008
batch_size = 256
samples = 2 ** 16
batches = samples // batch_size

# Differential Privacy parameters
# Choose preset: "high", "medium", or "low"
privacy_preset = "medium"
enable_dp = True  # Set to False to disable differential privacy

# Load privacy preset
dp_params = get_privacy_preset(privacy_preset)
print(f"Privacy preset: {privacy_preset}")
print(f"Description: {dp_params['description']}")
print(f"Parameters: l2_norm_clip={dp_params['l2_norm_clip']}, "
      f"noise_multiplier={dp_params['noise_multiplier']}, "
      f"target_epsilon={dp_params['target_epsilon']}")

In [ ]:
p_input = Input(shape=(p_len,), name="plaintext")
k_input = Input(shape=(k_len,), name="key")
c_input = Input(shape=(c_len,), name="ciphertext")

## Alice (Encryption Network)

In [ ]:
alice_inputs = [p_input, k_input]
x = Concatenate(axis=1, name="alice_concatenate")(alice_inputs)
x = Dense(units=(32), activation="relu", name="alice_dense")(x)
x = Reshape(target_shape=(32, 1), name="alice_reshape")(x)
x = Conv1D(filters=2, kernel_size=4, strides=1, padding="same", activation="relu", name="alice_conv1d_1")(x)
x = Conv1D(filters=4, kernel_size=2, strides=2, padding="same", activation="relu", name="alice_conv1d_2")(x)
x = Conv1D(filters=4, kernel_size=1, strides=1, padding="same", activation="relu", name="alice_conv1d_3")(x)
x = Conv1D(filters=1, kernel_size=1, strides=1, padding="same", activation="tanh", name="alice_conv1d_4")(x)
alice_outputs = Flatten(name="alice_flatten")(x)
alice = Model(inputs=alice_inputs, outputs=alice_outputs, name="alice")
alice.compile()

## Bob (Decryption Network)

In [ ]:
bob_inputs = [c_input, k_input]
x = Concatenate(axis=1, name="bob_concatenate")(bob_inputs)
x = Dense(units=(32), activation="relu", name="bob_dense")(x)
x = Reshape(target_shape=(32, 1), name="bob_reshape")(x)
x = Conv1D(filters=2, kernel_size=4, strides=1, padding="same", activation="relu", name="bob_conv1d_1")(x)
x = Conv1D(filters=4, kernel_size=2, strides=2, padding="same", activation="relu", name="bob_conv1d_2")(x)
x = Conv1D(filters=4, kernel_size=1, strides=1, padding="same", activation="relu", name="bob_conv1d_3")(x)
x = Conv1D(filters=1, kernel_size=1, strides=1, padding="same", activation="tanh", name="bob_conv1d_4")(x)
bob_outputs = Flatten(name="bob_flatten")(x)
bob = Model(inputs=bob_inputs, outputs=bob_outputs, name="bob")
bob.compile()

## Eve (Eavesdropper Network)

In [ ]:
eve_inputs = c_input
x = Dense(units=(32), activation="relu", name="eve_dense")(eve_inputs)
x = Reshape(target_shape=(32, 1), name="eve_reshape")(x)
x = Conv1D(filters=2, kernel_size=4, strides=1, padding="same", activation="relu", name="eve_conv1d_1")(x)
x = Conv1D(filters=4, kernel_size=2, strides=2, padding="same", activation="relu", name="eve_conv1d_2")(x)
x = Conv1D(filters=4, kernel_size=1, strides=1, padding="same", activation="relu", name="eve_conv1d_3")(x)
x = Conv1D(filters=1, kernel_size=1, strides=1, padding="same", activation="tanh", name="eve_conv1d_4")(x)
eve_outputs = Flatten(name="eve_flatten")(x)
eve = Model(inputs=eve_inputs, outputs=eve_outputs, name="eve")
eve.compile()

## Training Setup with Differential Privacy

In [ ]:
# l1 distance metric for loss
def l1_distance(a, b):
    a = (a + 1) / 2
    b = (b + 1) / 2
    return tf.reduce_mean(tf.reduce_sum(tf.abs(a - b), axis=-1))

# create training batch
def create_batch():
    p_batch = np.random.choice([-1, 1], size=(batch_size, p_len))
    k_batch = np.random.choice([-1, 1], size=(batch_size, k_len))
    return p_batch, k_batch

In [ ]:
# single forward pass for symbolic links
alice_output = alice([p_input, k_input])
bob_output = bob([alice_output, k_input])
eve_output = eve(alice_output)

# loss and metric functions
tn_eve_loss = l1_distance(p_input, eve_output)
tn_alice_bob_loss = l1_distance(p_input, bob_output) + tf.square(p_len / 2 - tn_eve_loss) / ((p_len / 2) ** 2)
tn_alice_bob_metric = l1_distance(p_input, bob_output)
tn_eve_metric = l1_distance(p_input, eve_output)

# create auxiliary training networks
tn_alice_bob = Model(inputs=[p_input, k_input], outputs=bob_output, name="tn_alice_bob")
tn_alice_bob.add_loss(tn_alice_bob_loss)
tn_alice_bob.add_metric(tn_alice_bob_metric, name="l1_distance")

tn_eve = Model(inputs=[p_input, k_input], outputs=eve_output, name="tn_eve")
tn_eve.add_loss(tn_eve_loss)
tn_eve.add_metric(tn_eve_metric, name="l1_distance")

# Setup optimizers with optional differential privacy
if enable_dp:
    # Create DP optimizers
    base_optimizer_ab = Adam(learning_rate=learning_rate)
    dp_optimizer_ab = DPOptimizer(
        optimizer=base_optimizer_ab,
        l2_norm_clip=dp_params['l2_norm_clip'],
        noise_multiplier=dp_params['noise_multiplier'],
        learning_rate=learning_rate
    )
    
    base_optimizer_eve = Adam(learning_rate=learning_rate)
    dp_optimizer_eve = DPOptimizer(
        optimizer=base_optimizer_eve,
        l2_norm_clip=dp_params['l2_norm_clip'],
        noise_multiplier=dp_params['noise_multiplier'],
        learning_rate=learning_rate
    )
    
    # Initialize privacy accountant
    privacy_accountant = PrivacyAccountant(
        noise_multiplier=dp_params['noise_multiplier'],
        batch_size=batch_size,
        num_samples=samples,
        delta=dp_params['target_delta']
    )
    
    print(f"Differential Privacy enabled with {privacy_preset} preset")
else:
    tn_alice_bob.compile(Adam(learning_rate=learning_rate))
    tn_eve.compile(Adam(learning_rate=learning_rate))
    print("Training without differential privacy")

alice.trainable = False

## Training Loop with DP-SGD

In [ ]:
bob_train_errors = []
eve_train_errors = []
privacy_spent = []

epoch = 0
above_threshold = True
start_time = time.time()

with tqdm(total=epochs*batches, desc="Training", unit="batch") as pbar:
    while epoch < epochs and above_threshold:
        for batch in range(batches):
            p_batch, k_batch = create_batch()
            
            if enable_dp:
                # Train Alice-Bob with DP-SGD
                with tf.GradientTape() as tape:
                    predictions = tn_alice_bob([p_batch, k_batch], training=True)
                    loss = tn_alice_bob.compiled_loss(None, predictions)
                    alice_bob_loss = loss
                
                # Compute gradients with DP
                grads_and_vars = dp_optimizer_ab.compute_gradients(
                    loss, tn_alice_bob.trainable_variables, tape
                )
                dp_optimizer_ab.apply_gradients(grads_and_vars)
                
                # Evaluate
                _, bob_error = tn_alice_bob.evaluate([p_batch, k_batch], verbose=0)
                bob_train_errors.append(bob_error)
                
                # Train Eve for 2 batches with DP-SGD
                for _ in range(2):
                    p_batch_eve, k_batch_eve = create_batch()
                    with tf.GradientTape() as tape_eve:
                        predictions_eve = tn_eve([p_batch_eve, k_batch_eve], training=True)
                        loss_eve = tn_eve.compiled_loss(None, predictions_eve)
                    
                    grads_and_vars_eve = dp_optimizer_eve.compute_gradients(
                        loss_eve, tn_eve.trainable_variables, tape_eve
                    )
                    dp_optimizer_eve.apply_gradients(grads_and_vars_eve)
                
                _, eve_error = tn_eve.evaluate([p_batch, k_batch], verbose=0)
                eve_train_errors.append(eve_error)
                
                # Update privacy accountant
                privacy_accountant.step()
                epsilon, delta = privacy_accountant.get_privacy_spent()
                privacy_spent.append(epsilon)
                
                # update progress bar
                pbar.set_postfix({
                    "alice_bob_loss": alice_bob_loss.numpy(),
                    "bob_error": bob_error,
                    "eve_error": eve_error,
                    "epsilon": f"{epsilon:.2f}"
                })
            else:
                # Standard training without DP
                tn_alice_bob.train_on_batch([p_batch, k_batch])
                alice_bob_loss, bob_error = tn_alice_bob.evaluate([p_batch, k_batch], verbose=0)
                bob_train_errors.append(bob_error)
                
                # train Eve for 2 batches
                for _ in range(2):
                    p_batch, k_batch = create_batch()
                    tn_eve.train_on_batch([p_batch, k_batch])
                _, eve_error = tn_eve.evaluate([p_batch, k_batch], verbose=0)
                eve_train_errors.append(eve_error)
                
                # update progress bar
                pbar.set_postfix({
                    "alice_bob_loss": alice_bob_loss,
                    "bob_error": bob_error,
                    "eve_error": eve_error
                })
            
            pbar.update()
            
            # exit if Alice and Bob loss is below threshold
            if alice_bob_loss < loss_threshold:
                print("Minimum loss threshold reached, exiting early")
                above_threshold = False
                break
        
        epoch += 1

total_time = time.strftime("%M:%S", time.gmtime(time.time() - start_time))
print(f"Training finished ({total_time})")

if enable_dp:
    final_epsilon, final_delta = privacy_accountant.get_privacy_spent()
    print(f"\nPrivacy spent: epsilon = {final_epsilon:.2f}, delta = {final_delta:.2e}")
    print(f"Target epsilon: {dp_params['target_epsilon']}")
    if final_epsilon <= dp_params['target_epsilon']:
        print("✓ Privacy budget satisfied!")
    else:
        print("⚠ Privacy budget exceeded. Consider:")
        print("  - Increasing noise_multiplier")
        print("  - Reducing number of epochs")
        print("  - Increasing batch size")

## Visualization

In [ ]:
# plot training errors
fig, axes = plt.subplots(1, 2 if enable_dp else 1, figsize=(16 if enable_dp else 8, 6))

if enable_dp:
    ax1, ax2 = axes
else:
    ax1 = axes

# Plot reconstruction errors
ax1.plot(bob_train_errors, label="Bob")
ax1.plot(eve_train_errors, label="Eve")
ax1.set_title(f"Symmetric model training errors{' (with DP)' if enable_dp else ''}")
ax1.set_xlabel("Batches")
ax1.set_ylabel(f"Bits wrong (of {p_len})")
ax1.set_yticks(np.arange(0, (p_len / 2) + 0.5, 0.5))
ax1.legend()

if enable_dp:
    # Plot privacy budget
    ax2.plot(privacy_spent, label="Epsilon", color="red")
    ax2.axhline(y=dp_params['target_epsilon'], color="orange", linestyle="--", label="Target epsilon")
    ax2.set_title("Privacy Budget Over Training")
    ax2.set_xlabel("Batches")
    ax2.set_ylabel("Epsilon (ε)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Models

In [ ]:
# # save models
# suffix = "_dp" if enable_dp else ""
# alice.save(f"models/symmetric/alice{suffix}.keras")
# bob.save(f"models/symmetric/bob{suffix}.keras")
# eve.save(f"models/symmetric/eve{suffix}.keras")
# print(f"Models saved with suffix: {suffix}")

# # load models
# alice = keras.models.load_model(f"models/symmetric/alice{suffix}.keras")
# bob = keras.models.load_model(f"models/symmetric/bob{suffix}.keras")
# eve = keras.models.load_model(f"models/symmetric/eve{suffix}.keras")

## Evaluation

In [ ]:
bob_eval_errors = []
eve_eval_errors = []

start_time = time.time()
with tqdm(total=batches, desc="Evaluation", unit="batch") as pbar:
    for batch in range(batches):
        p_batch, k_batch = create_batch()
        alice_bob_loss, bob_error = tn_alice_bob.evaluate([p_batch, k_batch], verbose=0)
        bob_eval_errors.append(bob_error)
        
        p_batch, k_batch = create_batch()
        _, eve_error = tn_eve.evaluate([p_batch, k_batch], verbose=0)
        eve_eval_errors.append(eve_error)
        
        # update progress bar
        pbar.set_postfix({"alice_bob_loss": alice_bob_loss, "bob_error": bob_error, "eve_error": eve_error})
        pbar.update()

total_time = time.strftime("%M:%S", time.gmtime(time.time() - start_time))
print(f"Evaluation finished ({total_time})")

In [ ]:
# plot evaluation errors
plt.figure(figsize=(8, 6))
plt.plot(bob_eval_errors, label="Bob")
plt.plot(eve_eval_errors, label="Eve")
plt.title(f"Symmetric model evaluation errors{' (with DP)' if enable_dp else ''}")
plt.xlabel("Batches")
plt.ylabel(f"Bits wrong (of {p_len})")
plt.yticks(np.arange(0, (p_len / 2) + 0.5, 0.5))
plt.legend()
plt.show()

## Text Encryption Demo

In [ ]:
# convert text of utf-8 characters to tensor
def text_to_tensor(text, p_len):
    # convert single utf-8 character to 8-bit binary list
    def char_to_binary(ch):
        return [int(bit) for bit in format(ord(ch), "08b")]
    
    binary = np.array([char_to_binary(ch) for ch in text]).flatten()
    # pad binary list to multiple of p_len
    pad = (p_len - len(binary) % p_len) % p_len
    tensor = np.concatenate([(binary * 2) - 1, np.zeros(pad)])
    return tensor, pad

# convert tensor to text of utf-8 characters
def tensor_to_text(tensor, pad):
    # convert 8-bit binary list to single utf-8 character
    def binary_to_char(binary):
        return chr(int("".join([str(bit) for bit in binary]), 2))
    
    binary = np.round((tensor + 1) / 2.0).astype("int").flatten()
    binarys = [binary[i: i + 8] for i in range(0, len(binary) - pad, 8)]
    return "".join(map(binary_to_char, binarys))

# perform symmetric encryption/decryption on text using trained models
def symmetric_encryption(plaintext):
    tensor, pad = text_to_tensor(plaintext, p_len)
    p_inputs = np.array(tensor).reshape(-1, p_len)
    k_inputs = np.random.choice([-1, 1], size=(len(p_inputs), k_len))
    
    alice_output = alice([p_inputs, k_inputs])
    bob_output = bob([alice_output, k_inputs])
    eve_output = eve(alice_output)
    
    ciphertext = tensor_to_text(alice_output, pad)
    plaintext_bob = tensor_to_text(bob_output, pad)
    plaintext_eve = tensor_to_text(eve_output, pad)
    
    return ciphertext, plaintext_bob, plaintext_eve

In [ ]:
plaintext = "Hello, World!"
ciphertext, plaintext_bob, plaintext_eve = symmetric_encryption(plaintext)
print(f"Plaintext: {plaintext}")
print(f"Ciphertext: {ciphertext}")
print(f"Plaintext (Bob): {plaintext_bob}")
print(f"Plaintext (Eve): {plaintext_eve}")

if enable_dp:
    print(f"\n[Differential Privacy: epsilon={final_epsilon:.2f}, delta={final_delta:.2e}]")